# 02. Модуль 2 — распознавание эмоций

Сравниваем представления аудио (waveform, лог-мел, HuBERT, латенты VAE) и функции потерь (CE, взвешенная CE, focal).
Главная метрика — **macro F1** (классы несбалансированы), дополнительно UA и WA.

In [ ]:
# Подключаем Google Drive, скачиваем код и ставим зависимости (~2 минуты)
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/tanyalebedeva/call-audio-analysis.git'
!test -d /content/call-audio-analysis || git clone -q $REPO /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
!mkdir -p $DATA $RUNS
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU не подключён: Среда выполнения → Сменить среду → T4 GPU'

## Представления аудио: одна функция потерь (CE)

In [ ]:
!python -m callaudio.train --config configs/emotion/mel_cnn.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/wave_cnn.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/hubert_frozen.yaml

In [ ]:
!python -m callaudio.train --config configs/emotion/hubert_finetune.yaml

Латенты VAE: сначала обучите `vae_dusha` в ноутбуке 03, потом запустите эту ячейку.

In [ ]:
!python -m callaudio.train --config configs/emotion/latent_vae.yaml

## Функции потерь против дисбаланса
Подставьте лучшую модель из таблицы выше (по умолчанию — мел-CNN, она быстрая).

In [ ]:
CFG = 'configs/emotion/mel_cnn.yaml'
for loss in ['weighted_ce', 'focal']:
    !python -m callaudio.train --config {CFG} --set loss.name={loss} out_dir={RUNS}/emo_mel_{loss}

## Оценка: чистая речь, шум, шум + очистка модулем 1

In [ ]:
import glob
ENHANCER = f'{RUNS}/sep_unet_sisdr/best.pt'   # лучшая модель модуля 1
for ckpt in sorted(glob.glob(f'{RUNS}/emo_*/best.pt')):
    !python -m callaudio.evaluate emotion --ckpt {ckpt}
    for snr in [0, 5]:
        !python -m callaudio.evaluate emotion --ckpt {ckpt} --noise-snr {snr}
        !python -m callaudio.evaluate emotion --ckpt {ckpt} --noise-snr {snr} --enhancer {ENHANCER}

In [ ]:
!python -m callaudio.report --runs $RUNS --out $RUNS/../reports
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{RUNS}/../reports/emotion.md', encoding='utf-8').read()))
display(Image(f'{RUNS}/emo_mel_ce/confusion.png'))

## Какой слой HuBERT лучше кодирует эмоции

In [ ]:
import json, matplotlib.pyplot as plt
for run in ['emo_hubert_frozen_ce', 'emo_hubert_ft_ce']:
    r = json.load(open(f'{RUNS}/{run}/results_emotion.json'))
    plt.plot(r['layer_importance'], marker='o', label=run)
plt.xlabel('слой (0 — выход CNN-кодировщика)'); plt.ylabel('вес слоя'); plt.legend(); plt.grid(alpha=.3)
plt.title('Обученные веса слоёв HuBERT'); plt.savefig(f'{RUNS}/../reports/hubert_layers.png', dpi=150); plt.show()

## Визуализация пространства эмбеддингов по эпохам

In [ ]:
for run in ['emo_mel_ce', 'emo_hubert_ft_ce', 'emo_wave_ce']:
    !python -m callaudio.viz --run {RUNS}/{run}
from IPython.display import Image
Image(f'{RUNS}/emo_mel_ce/viz/umap_evolution.gif')